In [3]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F
from databricks.sdk.runtime import dbutils

In [2]:
spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

In [4]:
dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")

print(f"Using catalog: {catalog}")

Using catalog: workspace


/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [5]:
dim_customer = spark.table(f"{catalog}.gold.dim_customer")
dim_product = spark.table(f"{catalog}.gold.dim_product")
dim_date = spark.table(f"{catalog}.gold.dim_date")
fact_order_item = spark.table(f"{catalog}.gold.fact_order_item")

print("dim_customer:", dim_customer.count())
print("dim_product:", dim_product.count())
print("dim_date:", dim_date.count())
print("fact_order_item:", fact_order_item.count())

dim_customer: 96097
dim_product: 32952
dim_date: 774
fact_order_item: 112642


In [6]:
checks = {
    "duplicate_customer_keys": (
        dim_customer
        .groupBy("customer_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),

    "duplicate_product_keys": (
        dim_product
        .groupBy("product_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),

    "duplicate_date_keys": (
        dim_date
        .groupBy("date_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),

    "duplicate_fact_keys": (
        fact_order_item
        .groupBy("order_id", "order_item_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),
}

for check, result in checks.items():
    print(f"{check}: {result}")

duplicate_customer_keys: 0
duplicate_product_keys: 0
duplicate_date_keys: 0
duplicate_fact_keys: 0


In [7]:
missing_customer_keys = (
    fact_order_item
    .join(dim_customer.select("customer_key"), on="customer_key", how="left_anti")
    .count()
)

missing_product_keys = (
    fact_order_item
    .join(dim_product.select("product_key"), on="product_key", how="left_anti")
    .count()
)

missing_date_keys = (
    fact_order_item
    .join(dim_date.select("date_key"), on="date_key", how="left_anti")
    .count()
)

print("Missing customer dimension keys:", missing_customer_keys)
print("Missing product dimension keys:", missing_product_keys)
print("Missing date dimension keys:", missing_date_keys)

Missing customer dimension keys: 0
Missing product dimension keys: 0
Missing date dimension keys: 0


In [9]:
fact_quality_checks = fact_order_item.agg(
    F.sum(
        F.when(F.col("customer_key").isNull(), 1).otherwise(0)
    ).alias("null_customer_key"),

    F.sum(
        F.when(F.col("product_key").isNull(), 1).otherwise(0)
    ).alias("null_product_key"),

    F.sum(
        F.when(F.col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_date_key"),

    F.sum(
        F.when(F.col("price") < 0, 1).otherwise(0)
    ).alias("negative_price"),

    F.sum(
        F.when(F.col("freight_value") < 0, 1).otherwise(0)
    ).alias("negative_freight"),

    F.sum(
        F.when(F.col("item_total") < 0, 1).otherwise(0)
    ).alias("negative_item_total")
)

fact_quality_checks.show(truncate=False)

+-----------------+----------------+-------------+--------------+----------------+-------------------+
|null_customer_key|null_product_key|null_date_key|negative_price|negative_freight|negative_item_total|
+-----------------+----------------+-------------+--------------+----------------+-------------------+
|0                |0               |0            |0             |0               |0                  |
+-----------------+----------------+-------------+--------------+----------------+-------------------+



In [10]:
unknown_member_checks = {
    "customer_unknown_rows": (
        dim_customer
        .filter(F.col("customer_key") == 0)
        .count()
    ),

    "product_unknown_rows": (
        dim_product
        .filter(F.col("product_key") == 0)
        .count()
    ),
}

for check, result in unknown_member_checks.items():
    print(f"{check}: {result}")

customer_unknown_rows: 1
product_unknown_rows: 1


In [12]:
fact_quality = fact_quality_checks.first()

validation_results = [
    ("duplicate_customer_keys", checks["duplicate_customer_keys"], 0),
    ("duplicate_product_keys", checks["duplicate_product_keys"], 0),
    ("duplicate_date_keys", checks["duplicate_date_keys"], 0),
    ("duplicate_fact_keys", checks["duplicate_fact_keys"], 0),

    ("missing_customer_keys", missing_customer_keys, 0),
    ("missing_product_keys", missing_product_keys, 0),
    ("missing_date_keys", missing_date_keys, 0),

    ("null_customer_key", fact_quality["null_customer_key"], 0),
    ("null_product_key", fact_quality["null_product_key"], 0),
    ("null_date_key", fact_quality["null_date_key"], 0),

    ("negative_price", fact_quality["negative_price"], 0),
    ("negative_freight", fact_quality["negative_freight"], 0),
    ("negative_item_total", fact_quality["negative_item_total"], 0),

    ("customer_unknown_rows", unknown_member_checks["customer_unknown_rows"], 1),
    ("product_unknown_rows", unknown_member_checks["product_unknown_rows"], 1),
]

validation_summary = spark.createDataFrame(
    validation_results,
    ["check_name", "actual_value", "expected_value"]
).withColumn(
    "status",
    F.when(
        F.col("actual_value") == F.col("expected_value"),
        F.lit("PASS")
    ).otherwise(F.lit("FAIL"))
)

validation_summary.show(30, truncate=False)

+-----------------------+------------+--------------+------+
|check_name             |actual_value|expected_value|status|
+-----------------------+------------+--------------+------+
|duplicate_customer_keys|0           |0             |PASS  |
|duplicate_product_keys |0           |0             |PASS  |
|duplicate_date_keys    |0           |0             |PASS  |
|duplicate_fact_keys    |0           |0             |PASS  |
|missing_customer_keys  |0           |0             |PASS  |
|missing_product_keys   |0           |0             |PASS  |
|missing_date_keys      |0           |0             |PASS  |
|null_customer_key      |0           |0             |PASS  |
|null_product_key       |0           |0             |PASS  |
|null_date_key          |0           |0             |PASS  |
|negative_price         |0           |0             |PASS  |
|negative_freight       |0           |0             |PASS  |
|negative_item_total    |0           |0             |PASS  |
|customer_unknown_rows  

In [13]:
failed_checks = (
    validation_summary
    .filter(F.col("status") == "FAIL")
)

failed_count = failed_checks.count()

if failed_count > 0:
    failed_checks.show(truncate=False)
    raise Exception(
        f"Gold validation failed: {failed_count} check(s) failed."
    )

print("Gold validation passed: all checks succeeded.")

Gold validation passed: all checks succeeded.
